# 5 A more complex neural network
In this final exercise, you will extend the concepts learned so far by defining and training a multi-layer neural network using PyTorch. This will allow you to understand how deeper architectures can model more complex relationships in the data compared to the single-layer models explored earlier.

### From before we have

In [ ]:
import matplotlib.pyplot as plt

import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import TensorDataset, DataLoader 
from torch.optim import SGD

import numpy as np

In [ ]:
def get_train_test():
    train_dataset = datasets.MNIST(root="data", train=True, download=True)
    test_dataset = datasets.MNIST(root="data", train=False, download=True)
    return train_dataset, test_dataset

In [ ]:
def preprocessing_pipeline(train_dataset):

    mu = 0.1307
    sigma = 0.3081

    # build the preprocessing transformation pipeline
    img_preprocessing_transformation_pipeline = transforms.Compose([
        transforms.RandomRotation(45),
        transforms.ToTensor(),                              # layer 1: from PIL images --> tensor
        transforms.Normalize(mean=[mu], std=[sigma])            # layer 2: normalize tensor images. They have to be in lists, otherwise that stupid shit doesn't work
    ])

    processed_dataset = []
    for img, lab in train_dataset:
        img_proc = img_preprocessing_transformation_pipeline(img)       # the image passes though the pipeline: converted into tensor and then normalized
        lab = torch.tensor(lab)         # I mean, since I'm already here, let's also convert the label into a tensor
        processed_dataset.append((img_proc, lab))
    
    return processed_dataset


In [ ]:
train_dataset, test_dataset = get_train_test()
processed_dataset = preprocessing_pipeline(train_dataset)

img = processed_dataset[0][0].shape
lab = processed_dataset[0][1].shape
print(img, lab)

for idx in range(2):
    print(processed_dataset[idx])

#### 5.1 Model definition: SimpleNN
You will define a simple feedforward neural network composed of multiple fully connected (Linear) layers. Each layer performs a linear transformation followed by a non-linear activation function, introducing flexibility into the model.

- Define the class: Define a class SimpleNN that inherits from nn. Module. In the constructor (__init__), specify three fully connected layers using nn. Linear.
    - The input layer should receive the flattened image of size 28 × 28 = 784, the first hidden layer (the professor is referring to the same layer, the input layer = first hidden layer, **WHICH IS WRONG** BUT THE PROFESSOR IS A DUMBASS) should produce 512 features,
    - the second hidden layer 256 features,
    - and the output layer should produce 10 outputs (corresponding to the number of classes to predict).

> SO:  
self.layer_1(784, 512)  
self.layer_2(512, 256)  
self.layer_3(256, 10)  

- Prepare the forward method: Implement the forward() method. First, flatten the input tensor (keeping the batch dimension). Then, apply the first and second layers followed by ReLU activations (**porco dio muori ratto di merda, sembra lo fai apposta a scrivere come nei quiz della patente ritardato: ciò che il ratto voleva dire è applica layer 1, ReLu, poi layer 2, ReLu, poi layer 3 e stop**), and finally use the output layer without any activation.

> SO:  
> 1. flat 28x28 --> produces 28x28 = 784  
> 2. ReLu
> 3. self.layer_1(784, 512)  
> 4. self.layer_2(512, 256)  
> 5. ReLu  
> 6. self.layer_3(256, 10)  

In [ ]:
# how to flatten a [1, 28, 28] matrix?
# FLATTEN = .reshape(-1)
# try it out

flat = processed_dataset[0][0].reshape(-1)
print(flat.shape)

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()

        self.layer_1 = nn.Linear(784, 512)
        self.relu = nn.ReLU()
        self.layer_2 = nn.Linear(512, 256)
        # ReLu again
        self.layer_3 = nn.Linear(256, 10)


    def forward(self, x):               # x is a batch of ONLY images, images have shape [1, 28, 28] which is already annoying, BUT the DataLoader makes everything worse because it becomes [batch_size, 1, 28, 28]
        # flatten the 28x28 matrix      x : [batch_size, 1, 28, 28] --> [batch_size, 28x28]
        # KEEP FIRST DIMENSION, COLLAPSE THE OTHER 3 IN A 28*28 = 784
        x = x.reshape(x.shape[0], 784)

        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.layer_3(x)
        return x


#### 5.2 Training setup
Once the model is defined, set up the components required for training:
- Prepare the data: Use a DataLoader to create mini-batches from your dataset, with a batch size of 1024. Enable shuffling for the training loader to ensure that batches are sampled differently at each epoch.

- Define the loss function and optimizer: Initialize the model and move it to the selected device
(CPU or GPU). Use the CrossEntropyLoss criterion, which is appropriate for multi-class classification tasks. Choose Stochastic Gradient Descent (SGD) as the optimizer with a learning rate of 0.01 and momentum of 0.9.

In [ ]:
# build TensorDataset (you don't need it, we already have tuples (data, correspnding label)) and DataLoader to build batches
# build object of the class SimpleNN
# build loss function
# build optimizer

def get_loader(processed_dataset, processed_test):
    data_loader = DataLoader(processed_dataset, batch_size=1024, shuffle=True)
    data_loader_test = DataLoader(processed_test, batch_size=1024, shuffle=True)
    return data_loader, data_loader_test


def training_setup():
    model = SimpleNN()
    loss_fn = nn.CrossEntropyLoss()
    optimizer = SGD(params=model.parameters(), lr=0.01, momentum=0.9)
    
    return model, loss_fn, optimizer

Create a validation function: Before starting the training process, define a small validation function to evaluate model accuracy on the test set. The function should:  
1. set the model to evaluation mode using model.eval();  

2. disable gradient computation with torch.no_grad() (**again the rat is bitch ass, you need to recall it in a with, hate that rat**) to save memory and computation time;  

3. iterate over the dataloader, obtain the model's predictions, and compare them with the proper labels to compute, at the end, the percentage of correctly classified samples.  

Use torch.max(outputs.data, 1) to obtain the predicted class for each sample --> the **rat** is a dick because you need to use:  
> **values, indices = torch.max(outputs.data, 1)**  
Fucking hate that guy

## THE EVALUATION DOESN'T HAVE SHIT IN IT AND IT PISSES ME OFF
> values, indices = torch.max(outputs.data, 1)  
- values  = the max logit for each sample (not very important here)
- **indices** = the index of the max logit = **predicted class** (0–9)

##### Why in evaluation: no no_grad(), no loss.backward(), no optimizer.step()?
a) Training loop = where the model learns  
Here you WANT:  
- gradients ON
- optimizer.zero_grad()
- loss.backward()
- optimizer.step()

**This is the loop that makes the model less dumb over time.**  

b) Evaluation loop = where you ONLY MEASURE performance  
Here you want to freeze the model:  
- You do not update weights (optimizer.step()).
- You do not accumulate gradients (loss.backward()).
- You still compute predictions (logits = model(x_batch)), but just to see how many are correct.

> You want a clean answer to:  
> **“Given the model as it is now, how good is it on unseen data?”**  

> - Model improves only in the training loop.
> - **Evaluation loop is just: “take snapshot of current brain, see how dumb/smart it is”, nothing more.**

In [ ]:
def validation(model, data_loader, eval_num_epochs = 1):
    
    # set the model into evaluation mode
    model.eval()

    with torch.no_grad():                       
        
        # EVALUATION loop
        for epoch in range(eval_num_epochs):    
            
            correct = 0
            total = 0
            print(f'Epoch: {epoch+1}')
            for x_batch, y_batch in data_loader:
                # optimizer.zero_grad() --> NO OPTIMIZER IN THE EVALUATION LOOP
                
                logits = model(x_batch)
                
                # loss = loss_fn(logits, y_batch) --> NO LOSS FUNCTION, WE DON'T WANT THE MODEL TO LEARN, JUST TO SEE HOW GOOD IT IS AS IT IS
                # loss.backward()

                # optimizer.step()

                raw_values, indices = torch.max(logits.data, 1)
                # this is an obscure object, I just know that values are the logits and indeces are the classes
                # --> ex. [-12.457, -33452.2, 20, ...] each element should be the raw probabiity (not all-positive
                # and between 0-1) of a picture of belonging to the class represented by the index
                # --> if you SoftMax -12.457 it might become 0.0005 and it's the probability of the image of belonging
                # to class 0, because that number is in index 0
                
                
                # count correct predictions and compute accuracy
                for idx in range(len(y_batch)):
                    if indices[idx] == y_batch[idx]:
                        correct += 1
                    total += 1
            
            accuracy = correct / total          # accuracy per epoch
            print(f"Accuracy of the plain model: {accuracy}")
    return accuracy


if __name__ == '__main__':
    train_dataset, test_dataset = get_train_test()
    processed_dataset = preprocessing_pipeline(train_dataset)
    data_loader, model, loss_fn, optimizer = training_setup(processed_dataset)
    validation(model, data_loader)

    

In [ ]:
# cleaned
def validation(model, data_loader, eval_num_epochs = 1):

    # set the model into evaluation mode
    model.eval()

    with torch.no_grad():                       # no gradients, juts for evaluation
        
        # EVALUATION loop
        for epoch in range(eval_num_epochs):    # which is usually one, the model doesn't improve over time in the evaluation
            
            correct = 0
            total = 0
            for x_batch, y_batch in data_loader:
                logits = model(x_batch)
                
                raw_values, indices = torch.max(logits.data, 1)
                
                # count correct predictions and compute accuracy
                for idx in range(len(y_batch)):
                    if indices[idx] == y_batch[idx]:
                        correct += 1
                    total += 1
            
            accuracy = correct / total          # accuracy per epoch
    
    return accuracy


if __name__ == '__main__':
    train_dataset, test_dataset = get_train_test()
    processed_dataset = preprocessing_pipeline(train_dataset)
    data_loader, model, loss_fn, optimizer = training_setup(processed_dataset)
    accuracy = validation(model, data_loader)
    print(f"Accuracy of the plain model: {accuracy}")

#### SO
We understood that the accuracy of the evaluation is 10%  
So basically the dumb model without learning from its mistake already guesses 10% of the labels, it's normal that it's so low, no worries!!

Let's see how much it improves once it actually learns something from its mistakes.

#### 5.3 Training loop
You will now implement the training loop for the model. This process should be similar to the one used in the linear model, but adapted for the classification task. After training the model for five epochs, use your validation function to compute and print the final test accuracy on the test set.

In [ ]:
# This is training accuracy
def training_and_validation(data_loader, optimizer, model, loss_fn, num_epochs = 1):
    
    model.train()

    # num_epochs = 5

    # train the model
    for epoch in range(num_epochs):
        for x_batch, y_batch in data_loader:
            optimizer.zero_grad()

            logits = model(x_batch)

            loss = loss_fn(logits, y_batch)

            loss.backward()

            optimizer.step()

        training_accuracy = validation(model, data_loader)      # not so sure about it
        print(f"Epoch: {epoch+1}")
        print(f"Accuracy: {training_accuracy}")
        print()
    
    return training_accuracy


In [ ]:
# use test accuracy
    # for the test
    # no optimizer
    # no loss function
    # no learning from mistakes
# FREEZE THE MODEL --> model.eval() and with torch.no_grad()

def test_and_validation(model, data_loader_test, num_epochs= 1):
    model.eval()

    with torch.no_grad():

        # I could have just recalled validation but I made practice in this way
        for epoch in range(num_epochs):
            count = 0
            tot = 0

            for x_batch, y_batch in data_loader_test:
                logits = model(x_batch)

                raw_values, indices = torch.max(logits.data, 1)

                for idx in range(len(indices)):
                    if indices[idx] == y_batch[idx]:
                        count += 1
                    tot += 1
            
            test_accuracy = count / tot
            
        
        return test_accuracy

In [ ]:
## clean it up

if __name__ == '__main__':
    train_dataset, test_dataset = get_train_test()
    processed_dataset = preprocessing_pipeline(train_dataset)
    processed_test = preprocessing_pipeline(test_dataset)

    data_loader, data_loader_test = get_loader(processed_dataset, processed_test)
    plain_accuracy = validation(model, data_loader)
    print(f"Accuracy of the plain model without training it: {plain_accuracy}")

    model, loss_fn, optimizer = training_setup()
    num_epochs = 5
    training_accuracy = training_and_validation(data_loader, optimizer, model, loss_fn, num_epochs)
    print(f"Accuracy of the trained model on training data: {training_accuracy}")

    test_accuracy = test_and_validation(model, data_loader_test)
    print(f'Test accuracy: {test_accuracy}')


# STILL DID NOT UNDERSTAND THE VALIDATION OR EVALUATION OR I DON'T FUCKING KNOW IT'S JUST A MESS AND THEY GET SWITCHED: FUCK THIS
Think of three buckets of data:
| Bucket    | What it’s for                                        | Does model learn from it?        |
|----------|------------------------------------------------------|----------------------------------|
| Train    | Teach the model                                      | ✅ yes (gradients + step)        |
| Validation | Check how good the model is while you’re building/tuning it | ❌ no                    |
| Test     | Final exam at the very end                           | ❌ no                            |

- Train = where you do:
```python
model.train()
loss.backward()
optimizer.step()
```

- Validation = where you do:
```python
model.eval()
with torch.no_grad():
    logits = model(x)
    # metrics only
```

- Test = same code as validation, but on the test set, and usually only once at the end.

- “evaluation” is the action:
“run the model on some dataset without training and compute metrics”.
- “validation” is a role of a dataset:
“this subset is used during development to pick models/hyperparams”.
- “test” is another role:
“this subset is only used at the end to report final performance”.

> You evaluate on the validation set, or you evaluate on the test set.


#### so evaluation is the action of running the model without learning from its mistakes and it uses a validation set, which is the subset used during the evaluation where the model doesn't learn and just gives a score?
### Evaluation = the action

**Evaluation = the action**  
“Run the model on some data, don’t update weights, compute a score (accuracy, loss, whatever).”  

Technically you can evaluate on:
- training set (training accuracy),
- validation set (validation accuracy),
- test set (test accuracy).

**Validation set = a subset of data**  
Its role is:
- data the model is not trained on,
- used during development to check how well it generalizes,
- used to compare/tune hyperparameters.

So:  
**evaluation** is the action,  
**validation set** is one of the places where you perform that action.


#### Why no learning in validation / test?

Because they answer a different question.  

- Training question:  
  “How do I change the weights to reduce the loss on this data?”

- Validation / test question:  
  “If I freeze the current weights and throw new data at the model,  
  how often is it right?”

If you let the model learn on validation/test (using gradients and optimizer):
- You’re no longer measuring generalization,
- You’re secretly training on that data,
- Your metrics lie (overly optimistic).



**“But in regression I used validation to tune hyperparameters…”**

That’s still true here. The generic roles are:
- Train: fit parameters (weights).
- Validation: choose hyperparameters / choose which model checkpoint is best.
- Test: one-shot final evaluation of the chosen model.

**For your neural net:**
- You could try:
  - different learning rates (0.01 vs 0.001),
  - different hidden sizes (256 vs 512),
  - different augmentation (with/without RandomRotation).
- For each choice (each run), you:
  - train on train set,
  - evaluate on validation set,
  - pick the hyperparameters that give best validation accuracy.

Then you freeze hyperparameters and only at the end:
- train one final time (train set),
- evaluate once on test set → final result.


### aahhhhh, I see, so I would have been way less confused if our model actually tried different hyperparameters, it confused me because we fixed all hyperparameters and we didn't performa gridsearch or something like this. So in general the evaluation that uses the validation set is used to tune the hyperparameters and give a score, but here the hyperparameters are fixed so it just gives a score?

In general, the typical workflow is:
1. Choose a set of hyperparameters (lr, hidden sizes, etc.).
2. Train on train set.
3. Evaluate on validation set → get a score.
4. Try another hyperparameter set → another score on validation.
5. Pick hyperparameters with best validation score.
6. Only at the end: train final model (sometimes retrain on train+val) and evaluate once on test.

So evaluation on validation is mainly for:
- tuning hyperparameters,
- picking the best model,
- tracking overfitting.

In this lab, they:
- fixed all hyperparameters (lr = 0.01, hidden sizes, etc.),
- didn’t ask you to try alternatives,
- and they didn’t even create a separate validation split: you have only train + test.

So:
- Your training_and_validation(...validation(model, train_loader)...)  
  is actually “evaluation on train set” → training accuracy.
- Your test_and_validation(model, data_loader_test)  
  is evaluation on test set → test accuracy.
- There is no real validation set in the usual sense.

Because hyperparameters are fixed, the “evaluation” here is doing just the second half of its usual job:
- It only gives a score at a given time (after each epoch, and at the end),
- but you’re not using that score to change hyperparameters.

So your mental summary:
- In theory:
  - evaluation on validation = used to tune hyperparameters + monitor generalization.
  - evaluation on test = final exam, one-shot report.
- In this lab:
  - hyperparameters are fixed → evaluation is only “how good is the model now?”, not “which hyperparameters win”.
  - they reused the word “validation” even though you’re mostly doing:
    - evaluation on train (training accuracy),
    - and evaluation on test (final accuracy).




---

---

# Complete compact model

In [ ]:
def get_train_test():
    train_dataset = datasets.MNIST(root="data", train=True, download=True)
    test_dataset = datasets.MNIST(root="data", train=False, download=True)
    return train_dataset, test_dataset


def preprocessing_pipeline(train_dataset):

    mu = 0.1307
    sigma = 0.3081

    # build the preprocessing transformation pipeline FOR IMAGES
    img_preprocessing_transformation_pipeline = transforms.Compose([
        transforms.RandomRotation(45),
        transforms.ToTensor(),                              # layer 1: from PIL images --> tensor
        transforms.Normalize(mean=[mu], std=[sigma])            # layer 2: normalize tensor images. They have to be in lists, otherwise that stupid shit doesn't work
    ])

    processed_dataset = []
    for img, lab in train_dataset:
        img_proc = img_preprocessing_transformation_pipeline(img)       # the image passes though the pipeline: converted into tensor and then normalized
        lab = torch.tensor(lab)         # I mean, since I'm already here, let's also convert the label into a tensor
        processed_dataset.append((img_proc, lab))
    
    return processed_dataset


class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()

        self.layer_1 = nn.Linear(784, 512)
        self.relu = nn.ReLU()
        self.layer_2 = nn.Linear(512, 256)
        # ReLu again
        self.layer_3 = nn.Linear(256, 10)


    def forward(self, x):               # x is a batch of ONLY images, images have shape [1, 28, 28] which is already annoying, BUT the DataLoader makes everything worse because it becomes [batch_size, 1, 28, 28]
        # flatten the 28x28 matrix      x : [batch_size, 1, 28, 28] --> [batch_size, 28x28]
        # KEEP FIRST DIMENSION, COLLAPSE THE OTHER 3 IN A 28*28 = 784
        x = x.reshape(x.shape[0], 784)

        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.layer_3(x)
        return x


def get_loader(processed_dataset, processed_test):
    data_loader = DataLoader(processed_dataset, batch_size=1024, shuffle=True)
    data_loader_test = DataLoader(processed_test, batch_size=1024, shuffle=True)
    return data_loader, data_loader_test


def training_setup():
    model = SimpleNN()
    loss_fn = nn.CrossEntropyLoss()     # automatically performs logSoftmax --> convert raw logits into probabilities
    optimizer = SGD(params=model.parameters(), lr=0.01, momentum=0.9)
    
    return model, loss_fn, optimizer


def evaluation(model, data_loader, eval_num_epochs = 1):

    # set the model into evaluation mode
    model.eval()

    with torch.no_grad():                       # no gradients, juts for evaluation
        
        # EVALUATION loop
            # makes no sense: the number of epochs in evaluation is always one,
            # the model doesn't improve over time in the evaluation, so it makes no sense to use more than one epoch
            # still use one epoch even if you were to tune hyperparameters

        for epoch in range(eval_num_epochs): 
            
            correct = 0
            total = 0
            for x_batch, y_batch in data_loader:
                logits = model(x_batch)
                
                raw_values, indices = torch.max(logits.data, 1)
                
                # count correct predictions and compute accuracy
                for idx in range(len(y_batch)):
                    if indices[idx] == y_batch[idx]:
                        correct += 1
                    total += 1
            
            accuracy = correct / total          # accuracy per epoch
    
    return accuracy


def training_and_evaluation(data_loader, optimizer, model, loss_fn, num_epochs = 1):
    
    model.train()

    # num_epochs = 5

    # train the model
    for epoch in range(num_epochs):
        for x_batch, y_batch in data_loader:
            optimizer.zero_grad()

            logits = model(x_batch)

            loss = loss_fn(logits, y_batch)

            loss.backward()

            optimizer.step()

        training_accuracy = evaluation(model, data_loader)
        print(f"Epoch: {epoch+1}")
        print(f"Accuracy: {training_accuracy}")
        print()
    
    return training_accuracy


def test_and_evaluation(model, data_loader_test, num_epochs= 1):
    # set the model into evaluation mode because we don't want it to learn from the mistakes
    model.eval()

    with torch.no_grad():

        # I could have just recalled validation but I want to practice
        # doesn't make sense to do more than one epoch, the model isn't learning
        for epoch in range(num_epochs):
            count = 0
            tot = 0

            for x_batch, y_batch in data_loader_test:
                logits = model(x_batch)

                raw_values, indices = torch.max(logits.data, 1)

                for idx in range(len(indices)):
                    if indices[idx] == y_batch[idx]:
                        count += 1
                    tot += 1
            
            test_accuracy = count / tot
            
        
        return test_accuracy




##############################################################################################################################################





if __name__ == '__main__':
    train_dataset, test_dataset = get_train_test()
    processed_dataset = preprocessing_pipeline(train_dataset)
    processed_test = preprocessing_pipeline(test_dataset)

    data_loader, data_loader_test = get_loader(processed_dataset, processed_test)
    model, loss_fn, optimizer = training_setup()
    
    plain_accuracy = evaluation(model, data_loader)
    print(f"Accuracy of the plain model without training it: {plain_accuracy}")

    num_epochs = 5
    training_accuracy = training_and_evaluation(data_loader, optimizer, model, loss_fn, num_epochs)
    print(f"Accuracy of the trained model on training data: {training_accuracy}")

    test_accuracy = test_and_evaluation(model, data_loader_test)
    print(f'Test accuracy: {test_accuracy}')
